In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── 1) Load pkl payload (true per-participant segment embeddings + all labels) ─────
with open("deep-prep-ai-audio-embeddings.pkl", "rb") as f:
    payload = pickle.load(f)

emb_list_raw = payload["transcript_embeddings"]
prosody_list_raw = payload["prosody_features"]

LABEL_COLS = [
    "interview_score",
    "overall_personality",
    "answer_score",
    "speaking_skills",
    "agreeableness",
    "conscientiousness",
    "neuroticism",
    "openness",
    "confidence_score",
]

def to_float32_label(col_values):
    return pd.to_numeric(pd.Series(col_values), errors="coerce").to_numpy(dtype=np.float32)

Y = np.stack([to_float32_label(payload[c]) for c in LABEL_COLS], axis=1)  # (N, num_targets)


def parse_embedding_entry(entry):
    if isinstance(entry, np.ndarray):
        arr = entry.astype(np.float32, copy=False)
    elif isinstance(entry, (list, tuple)):
        arr = np.array(entry, dtype=np.float32)
    elif isinstance(entry, str):
        cleaned = (
            entry.replace("\n", " ")
            .replace("[", " ")
            .replace("]", " ")
            .replace(",", " ")
        )
        flat = np.fromstring(cleaned, sep=" ", dtype=np.float32)
        if flat.size == 0:
            return None
        arr = flat
    else:
        return None

    if arr.ndim == 0:
        return None
    if arr.ndim > 2:
        return None
    return arr


parsed = [parse_embedding_entry(entry) for entry in emb_list_raw]

# Infer embedding dimension from valid 2D entries; fallback to MiniLM dim.
embed_dim_candidates = [arr.shape[1] for arr in parsed if isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[1] > 0]
embed_dim = int(embed_dim_candidates[0]) if len(embed_dim_candidates) > 0 else 384

# Convert any 1D parsed vectors into (n_segments, embed_dim) if possible.
emb_list = []
for arr in parsed:
    if arr is None:
        emb_list.append(None)
        continue

    if arr.ndim == 2:
        if arr.shape[1] != embed_dim:
            emb_list.append(None)
        else:
            emb_list.append(arr.astype(np.float32, copy=False))
    else:
        if arr.size % embed_dim != 0:
            emb_list.append(None)
        else:
            emb_list.append(arr.reshape(-1, embed_dim).astype(np.float32, copy=False))

N = min(len(emb_list), len(Y), len(prosody_list_raw))
if N == 0:
    raise ValueError("No rows found in embeddings/labels.")

emb_list = emb_list[:N]
Y = Y[:N]
prosody_list_raw = prosody_list_raw[:N]

MAX_SEQ_LEN = 64
num_targets = Y.shape[1]

# X_text / X_prosody: aligned segment sequences; shared padding mask
PROSODY_DIM = 13
X = np.zeros((N, MAX_SEQ_LEN, embed_dim), dtype=np.float32)
X_prosody = np.zeros((N, MAX_SEQ_LEN, PROSODY_DIM), dtype=np.float32)
key_padding_mask = np.ones((N, MAX_SEQ_LEN), dtype=bool)

valid_modal_rows = np.zeros(N, dtype=bool)
for i, arr in enumerate(emb_list):
    if arr is None or arr.ndim != 2 or arr.shape[1] != embed_dim:
        continue
    p = np.asarray(prosody_list_raw[i], dtype=np.float32)
    if p.ndim != 2 or p.shape[1] != PROSODY_DIM or p.shape[0] != arr.shape[0]:
        continue
    if not np.isfinite(p).all():
        continue
    n = min(arr.shape[0], MAX_SEQ_LEN)
    if n > 0:
        X[i, :n, :] = arr[:n, :]
        X_prosody[i, :n, :] = p[:n, :]
        key_padding_mask[i, :n] = False
        valid_modal_rows[i] = True

valid_rows = np.isfinite(Y).all(axis=1) & valid_modal_rows
X = X[valid_rows]
X_prosody = X_prosody[valid_rows]
key_padding_mask = key_padding_mask[valid_rows]
Y = Y[valid_rows]

if len(Y) == 0:
    raise ValueError("No valid rows after filtering. Check PKL export and label availability.")

idx = np.arange(len(Y))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train, X_val = X[train_idx], X[val_idx]
X_prosody_train, X_prosody_val = X_prosody[train_idx], X_prosody[val_idx]
mask_train, mask_val = key_padding_mask[train_idx], key_padding_mask[val_idx]
y_train, y_val = Y[train_idx], Y[val_idx]

print(f"Loaded {N} rows from PKL, using {len(Y)} valid rows after filtering.")
print(
    f"X_text {X.shape}, X_prosody {X_prosody.shape}, Y {Y.shape}, embed_dim={embed_dim}"
)

# ── 2) Dataset (text + prosody modalities) ─────────────────────────────────────
class MultimodalSeqDataset(Dataset):
    def __init__(self, X_text, X_prosody, key_padding_mask, y):
        self.X_text = torch.tensor(X_text, dtype=torch.float32)
        self.X_prosody = torch.tensor(X_prosody, dtype=torch.float32)
        self.key_padding_mask = torch.tensor(key_padding_mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_text[idx],
            self.X_prosody[idx],
            self.key_padding_mask[idx],
            self.y[idx],
        )

train_ds = MultimodalSeqDataset(X_train, X_prosody_train, mask_train, y_train)
val_ds = MultimodalSeqDataset(X_val, X_prosody_val, mask_val, y_val)

BATCH_SIZE = 16
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ── 3) Model: separate encoders (text vs prosody) + fusion + head ───────────
class ModalitySeqEncoder(nn.Module):
    """Transformer over one modality's segment features -> CLS vector (d_model,)."""

    def __init__(
        self,
        input_dim,
        max_seq_len,
        d_model=128,
        nhead=4,
        num_layers=2,
        dropout=0.1,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_len + 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, key_padding_mask):
        b, seq_len, _ = x.shape
        x = self.input_proj(x)
        cls = self.cls_token.expand(b, 1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, : seq_len + 1, :]
        cls_mask = torch.zeros((b, 1), dtype=torch.bool, device=x.device)
        full_mask = torch.cat([cls_mask, key_padding_mask], dim=1)
        x = self.encoder(x, src_key_padding_mask=full_mask)
        return x[:, 0, :]


class TwoTokenFusion(nn.Module):
    """Self-attention over [text_vec, audio_vec] then mean-pool."""

    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, text_vec, audio_vec):
        x = torch.stack([text_vec, audio_vec], dim=1)
        x = self.encoder(x)
        return x.mean(dim=1)


class MultimodalTransformerRegressor(nn.Module):
    def __init__(
        self,
        text_dim,
        prosody_dim,
        max_seq_len,
        num_targets,
        d_model=128,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        fusion="concat",
    ):
        super().__init__()
        self.fusion = fusion
        self.text_encoder = ModalitySeqEncoder(
            text_dim, max_seq_len, d_model, nhead, num_layers, dropout
        )
        self.audio_encoder = ModalitySeqEncoder(
            prosody_dim, max_seq_len, d_model, nhead, num_layers, dropout
        )
        if fusion == "concat":
            self.fuse_proj = nn.Sequential(
                nn.Linear(2 * d_model, d_model),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.two_token_fusion = None
        elif fusion == "attention":
            self.two_token_fusion = TwoTokenFusion(d_model, nhead, dropout)
            self.fuse_proj = None
        else:
            raise ValueError('fusion must be "concat" or "attention"')

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_targets),
        )

    def forward(self, x_text, key_padding_mask, x_prosody):
        text_repr = self.text_encoder(x_text, key_padding_mask)
        audio_repr = self.audio_encoder(x_prosody, key_padding_mask)
        if self.fusion == "concat":
            fused = self.fuse_proj(torch.cat([text_repr, audio_repr], dim=-1))
        else:
            fused = self.two_token_fusion(text_repr, audio_repr)
        return self.head(fused)


# Stronger regularization: reduce train/val gap (sweep WD: 0, 1e-5, 1e-4, 1e-3; dropout ~0.2–0.3)
DROPOUT = 0.25
WEIGHT_DECAY = 1e-3
# "concat": Linear(2*d_model -> d_model); "attention": 2-token TransformerEncoder + mean pool
FUSION = "concat"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalTransformerRegressor(
    text_dim=embed_dim,
    prosody_dim=PROSODY_DIM,
    max_seq_len=MAX_SEQ_LEN,
    num_targets=num_targets,
    d_model=128,
    nhead=4,
    num_layers=2,
    dropout=DROPOUT,
    fusion=FUSION,
).to(device)
print(model)
print("Targets:", LABEL_COLS)
print("Fusion:", FUSION)

# ── 4) Training loop (fixed LR — no ReduceLROnPlateau) ─────────────────────────
LR = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()
GRAD_CLIP_NORM = 1.0


def run_epoch(dl, train=True):
    model.train(train)
    total_loss = 0.0

    with torch.set_grad_enabled(train):
        for X_batch, X_prosody_batch, mask_batch, y_batch in dl:
            X_batch = X_batch.to(device)
            X_prosody_batch = X_prosody_batch.to(device)
            mask_batch = mask_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch, mask_batch, X_prosody_batch)
            loss = loss_fn(preds, y_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            total_loss += loss.item() * len(y_batch)

    return total_loss / len(dl.dataset)


EPOCHS = 30
MIN_EPOCHS_BEFORE_STOP = 15
PATIENCE = 8
MIN_DELTA = 1e-4

best_val_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_dl, train=True)
    val_loss = run_epoch(val_dl, train=False)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    elif epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
        epochs_no_improve += 1

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:3d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f} | lr: {LR:.2e} (fixed)"
        )

    if epoch + 1 >= MIN_EPOCHS_BEFORE_STOP and epochs_no_improve >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch+1}. Best val MSE {best_val_loss:.4f} at epoch {best_epoch}."
        )
        break

if best_state is not None:
    device = next(model.parameters()).device
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})


Loaded 2011 rows from PKL, using 1996 valid rows after filtering.
X_text (1996, 64, 384), X_prosody (1996, 64, 13), Y (1996, 9), embed_dim=384
MultimodalTransformerRegressor(
  (text_encoder): ModalitySeqEncoder(
    (input_proj): Linear(in_features=384, out_features=128, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.25, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.25, inplace=False)
          (dropout2): Dropout(p=0.25, inplace=False)


/Users/rayyanzaid/Desktop/School/CSCI-566-DeepLearning/CSCI-566-Course-Project-DeepPrep-AI/csci-566-project-venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


In [2]:
# Save model so I don't have to retrain every time for the Flask server inference

torch.save(model.state_dict(), "multimodal_transformer_audio_regressor.pth")